In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import sqlite3
import json
import re

# 1. Load page
url = "https://en.wikipedia.org/wiki/List_of_highest-grossing_films"
headers = {"User-Agent": "Mozilla/5.0"}

response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, "html.parser")

print("Loaded:", soup.title.text)

# 2. Find main table
table = soup.find("table", class_="wikitable")
rows = table.find_all("tr")

# 3. Get headers
header_cells = rows[0].find_all(["th", "td"])
headers = [h.get_text(strip=True) for h in header_cells]

print("Headers:", headers)

# 4. Find column indexes dynamically
def find_col(name):
    for i, h in enumerate(headers):
        if name.lower() in h.lower():
            return i
    return None

title_idx = find_col("title")
gross_idx = find_col("gross")
year_idx = find_col("year")

print("Indexes:", title_idx, gross_idx, year_idx)

# 5. Cleaning functions
def clean_text(text):
    if not text:
        return None
    text = re.sub(r"\[[^\]]*\]", "", text)
    text = text.replace("†", "")
    text = text.replace("\xa0", " ")
    return text.strip()

def extract_money(text):
    match = re.search(r"\$[\d,]+", text)
    return match.group() if match else None

def extract_year(text):
    match = re.search(r"\d{4}", text)
    return int(match.group()) if match else None

# 6. Parse data safely
films = []

for row in rows[1:]:
    cols = row.find_all(["td", "th"])  # ← ВОТ ЭТО КЛЮЧЕВОЕ
    
    if len(cols) < 5:
        continue

    title = clean_text(cols[title_idx].get_text(" ", strip=True))
    gross = extract_money(cols[gross_idx].get_text(" ", strip=True))
    year = extract_year(cols[year_idx].get_text(" ", strip=True))

    if title and gross:
        films.append({
            "title": title,
            "release_year": year,
            "director": "Unknown",
            "box_office": gross,
            "country": "Unknown"
        })
# 7. DataFrame
df = pd.DataFrame(films)
print("\nPreview:")
print(df.head(10))

# 8. Save to SQLite
conn = sqlite3.connect("films.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS films (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT,
    release_year INTEGER,
    director TEXT,
    box_office TEXT,
    country TEXT
)
""")

cursor.execute("DELETE FROM films")

for _, row in df.iterrows():
    cursor.execute("""
    INSERT INTO films (title, release_year, director, box_office, country)
    VALUES (?, ?, ?, ?, ?)
    """, (
        row["title"],
        row["release_year"],
        row["director"],
        row["box_office"],
        row["country"]
    ))

conn.commit()

# 9. Export JSON
df.to_json("films.json", orient="records", indent=4, force_ascii=False)

conn.close()

print("\nDONE ✅")
print("files created: films.db + films.json")

Loaded: List of highest-grossing films - Wikipedia
Headers: ['Rank', 'Peak', 'Title', 'Worldwide gross', 'Year', 'Ref']
Indexes: 2 3 4

Preview:
                          title  release_year director      box_office  \
0                        Avatar          2009  Unknown  $2,923,710,708   
1             Avengers: Endgame          2019  Unknown  $2,797,501,328   
2      Avatar: The Way of Water          2022  Unknown  $2,334,484,620   
3                       Titanic          1997  Unknown  $2,257,906,828   
4                      Ne Zha 2          2025  Unknown  $2,215,690,000   
5  Star Wars: The Force Awakens          2015  Unknown  $2,068,223,624   
6        Avengers: Infinity War          2018  Unknown  $2,048,359,754   
7       Spider-Man: No Way Home          2021  Unknown  $1,922,598,800   
8                    Zootopia 2          2025  Unknown  $1,866,641,191   
9                  Inside Out 2          2024  Unknown  $1,698,863,816   

   country  
0  Unknown  
1  Unknown  
2